<a href="https://colab.research.google.com/github/jwcunhagit/phd_ufpe/blob/ai-x-ethical-aspects/toolkit_bias_fairness_p2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#🛠️ Toolkit de Bias e Fairness em IA: Teoria e Prática (Parte 2/3)

> Material produzido para a disciplina 'Ética em Inteligência Artificial' ministrada pelo Prof. Geber Ramalho no Mestrado em Ciência da Computação do Centro de Informática da Universidade Federal de Pernambuco (CIn - UFPE)

Projeto finalizado e entregue em 16/12/25.

Discentes:

* Letícia Leitão ([github](https://github.com/leticialevleitao)/[linkedin](https://www.linkedin.com/in/leticialevleitao/))
* Renato Samico ([github](https://github.com/dsnato)/[linkedin](https://))

##Introdução
Antes de avançarmos para atividade prática, é fundamental entender como os vieses em IA se manifestam em cenários reais. Neste notebook, você irá explorar um caso emblemático que moldou o campo de Fairness em Inteligência Artificial, além de observar 3 ferramentas práticas recomendadas na literatura (AIF 360, Fairlearn e What-if Tool).

##Estudo de Caso com Dataset Real
###Caso COMPAS - Justiça Criminal
**O Problema:** Sistema usado nos EUA para prever reincidência criminal que apresentava viés racial significativo.

**O que você verá:**

- Como dados históricos de policiamento enviesado afetam algoritmos

- Métricas de fairness aplicadas ao sistema judiciário

- Consequências reais: pessoas negras recebiam scores 45% mais altos

**Conceitos-chave aprendidos:**

- False Positive Rate (FPR) desproporcional

- Viés de medição através de proxies

- Impacto social de sistemas algorítmicos

###Pronto para começar?
Nas células a seguir, você encontrará:

- Código completo para análise do caso COMPAS

- Simulação usando AIF 360 e Fairlearn

- What-if Tool: explicação e desafio


### Passo 1: Carregando o dataset

Primeiro, vamos carregar o dataset original do COMPAS, que contém as informações sobre os réus e suas pontuações de risco. Este é o ponto de partida para nossa análise.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Carrega o dataset COMPAS
compas_df = pd.read_csv('https://raw.githubusercontent.com/propublica/compas-analysis/master/compas-scores-two-years.csv')
print("\nDataset COMPAS original carregado com sucesso!")
print("\nPrimeiras 5 linhas\n")
display(compas_df.head())


Dataset COMPAS original carregado com sucesso!

Primeiras 5 linhas



,id,name,first,last,compas_screening_date,sex,dob,age,age_cat,race,...,v_decile_score,v_score_text,v_screening_date,in_custody,out_custody,priors_count.1,start,end,event,two_year_recid
0,1,miguel hernandez,miguel,hernandez,2013-08-14,Male,1947-04-18,69,Greater than 45,Other,...,1,Low,2013-08-14,2014-07-07,2014-07-14,0,0,327,0,0
1,3,kevon dixon,kevon,dixon,2013-01-27,Male,1982-01-22,34,25 - 45,African-American,...,1,Low,2013-01-27,2013-01-26,2013-02-05,0,9,159,1,1
2,4,ed philo,ed,philo,2013-04-14,Male,1991-05-14,24,Less than 25,African-American,...,3,Low,2013-04-14,2013-06-16,2013-06-16,4,0,63,0,1
3,5,marcu brown,marcu,brown,2013-01-13,Male,1993-01-21,23,Less than 25,African-American,...,6,Medium,2013-01-13,NaN,NaN,1,0,1174,0,0
4,6,bouthy pierrelouis,bouthy,pierrelouis,2013-03-26,Male,1973-01-22,43,25 - 45,Other,...,1,Low,2013-03-26,NaN,NaN,2,0,1102,0,0


### Passo 2: Observando o viés inicial

Um dos principais achados da análise do COMPAS foi que pessoas afro-americanas recebiam pontuações de risco mais altas. Vamos confirmar isso olhando a média do `decile_score` (pontuação de risco) para os dois grupos raciais mais representados no dataset: African-American e Caucasian.

In [ ]:
# Filtra o dataset para as raças 'African-American' e 'Caucasian' e remove valores ausentes no decile_score
df_filtered_races = compas_df[
    (compas_df['race'] == 'African-American') |
    (compas_df['race'] == 'Caucasian')
].dropna(subset=['decile_score'])

# Calcula o score decile médio por raça
print("Score decile médio por raça:\n")
print(df_filtered_races.groupby('race')['decile_score'].mean())

print("\nIsso nos mostra que, em média, indivíduos 'African-American' (negros) receberam scores de risco mais altos do que os Caucasianos (brancos), um indício claro do viés\n")

Score decile médio por raça:

race
African-American    5.368777
Caucasian           3.735126
Name: decile_score, dtype: float64

Isso nos mostra que, em média, indivíduos 'African-American' (negros) receberam scores de risco mais altos do que os Caucasianos (brancos), um indício claro do viés



### Passo 3: Preparando o dataset para melhor compreensão

Para facilitar a leitura e o entendimento das próximas análises, vamos criar uma cópia do nosso dataset e renomear algumas colunas-chave para o português. Assim, os termos ficarão mais claros para todos.

In [ ]:
compas_df_pt = compas_df.copy() # Criamos uma cópia para não alterar o dataset original
compas_df_pt = compas_df_pt.rename(columns={
    'race': 'raca',
    'decile_score': 'score_decil',
    'age': 'idade',
    'sex': 'sexo',
    'priors_count': 'contagem_antecedentes',
    'two_year_recid': 'reincidencia_dois_anos'
})

print("Colunas traduzidas com sucesso! Exemplo das novas colunas:")
display(compas_df_pt[['raca', 'score_decil', 'idade', 'sexo', 'reincidencia_dois_anos']].head())

Colunas traduzidas com sucesso! Exemplo das novas colunas:


,raca,score_decil,idade,sexo,reincidencia_dois_anos
0,Other,1,69,Male,0
1,African-American,3,34,Male,1
2,African-American,4,24,Male,1
3,African-American,8,23,Male,0
4,Other,1,43,Male,0


### Passo 4: Análise exploratória de dados (EDA)

Agora que temos o dataset com as colunas em português, vamos fazer uma análise exploratória mais aprofundada para entender melhor os dados e visualizar o viés de forma clara

#### 4.1. Informações gerais do dataset

Vamos ver um resumo da estrutura do nosso conjunto de dados, como o número total de registros (pessoas), quantas características (colunas) temos e qual o tipo de informação que cada coluna contém. Isso também nos ajuda a verificar se há dados faltando.

In [ ]:
print("\nInformações gerais do dataset COMPAS (em português)\n")
compas_df_pt.info()
print("\nO que aprendemos aqui: Veja que temos 7214 registros (pessoas), várias colunas com diferentes tipos de dados e que não há dados faltantes")


Informações gerais do dataset COMPAS (em português)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7214 entries, 0 to 7213
Data columns (total 53 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   id                       7214 non-null   int64  
 1   name                     7214 non-null   object 
 2   first                    7214 non-null   object 
 3   last                     7214 non-null   object 
 4   compas_screening_date    7214 non-null   object 
 5   sexo                     7214 non-null   object 
 6   dob                      7214 non-null   object 
 7   idade                    7214 non-null   int64  
 8   age_cat                  7214 non-null   object 
 9   raca                     7214 non-null   object 
 10  juv_fel_count            7214 non-null   int64  
 11  score_decil              7214 non-null   int64  
 12  juv_misd_count           7214 non-null   int64  
 13  juv_other_count         

#### 4.2. Estatísticas do Score Decil de Risco

O `score_decil` é a pontuação principal que o sistema COMPAS atribui, variando de 1 (menor risco) a 10 (maior risco). Vamos ver as estatísticas básicas dessa pontuação: a média, o valor mínimo, o valor máximo e como os scores se distribuem em quartis.

In [ ]:
print("\nEstatísticas do Score Decil de Risco\n")
print(compas_df_pt['score_decil'].describe())
print("\nO que aprendemos aqui? A 'mean' (média) nos dá uma ideia do risco médio geral no dataset. Os valores de 25%, 50% (mediana) e 75% nos mostram como as pontuações estão distribuídas. Por exemplo, 50% das pessoas têm um 'score_decil' igual ou menor que 4.")


Estatísticas do Score Decil de Risco

count    7214.000000
mean        4.509565
std         2.856396
min         1.000000
25%         2.000000
50%         4.000000
75%         7.000000
max        10.000000
Name: score_decil, dtype: float64

O que aprendemos aqui? A 'mean' (média) nos dá uma ideia do risco médio geral no dataset. Os valores de 25%, 50% (mediana) e 75% nos mostram como as pontuações estão distribuídas. Por exemplo, 50% das pessoas têm um 'score_decil' igual ou menor que 4.


#### 4.3. Distribuição dos Indivíduos por Raça

É importante saber quantos indivíduos de cada grupo racial estão presentes no nosso dataset. Isso nos ajuda a entender a proporção de cada grupo e a representatividade deles na análise de viés.

In [ ]:
print("\nDistribuição dos Indivíduos por Raça\n")
print(compas_df_pt['raca'].value_counts())
print("\nO que aprendemos aqui? Observamos que os grupos 'African-American' (negros) e 'Caucasian' (brancos) são os mais numerosos. Essa informação é fundamental, pois o viés tende a se manifestar de forma diferente entre os grupos, e a representatividade importa.")


Distribuição dos Indivíduos por Raça

raca
African-American    3696
Caucasian           2454
Hispanic             637
Other                377
Asian                 32
Native American       18
Name: count, dtype: int64

O que aprendemos aqui? Observamos que os grupos 'African-American' (negros) e 'Caucasian' (brancos) são os mais numerosos. Essa informação é fundamental, pois o viés tende a se manifestar de forma diferente entre os grupos, e a representatividade importa.


##Algoritmo usando AIF360 (IBM) - Detecção de Viés em Empréstimos

In [ ]:
!pip install aif360

In [ ]:
import pandas as pd
import numpy as np
from aif360.datasets import BinaryLabelDataset
from aif360.metrics import BinaryLabelDatasetMetric
from aif360.algorithms.preprocessing import Reweighing

# Simular dados de empréstimos bancários
# Atributos: renda, score_credito, idade (1=aprovado, 0=negado)
np.random.seed(42)
n_samples = 200

# Gerar dados sintéticos
dados = np.random.randn(n_samples, 3)
labels = (dados[:, 0] + dados[:, 1] * 0.5 + np.random.randn(n_samples) * 0.2 > 0).astype(int)

# Adicionar atributo sensível (gênero: 0=feminino, 1=masculino)
# Introduzir viés: mulheres têm menor chance de aprovação
genero = np.random.randint(0, 2, n_samples)
viés = np.where(genero == 0, -0.3, 0.2)  # Penaliza mulheres
labels_justos = (dados[:, 0] + dados[:, 1] * 0.5 + np.random.randn(n_samples) * 0.2 + viés > 0).astype(int)

# Criar dataset com atributos sensíveis
dataset = BinaryLabelDataset(
    df=pd.DataFrame({
        'renda': dados[:, 0],
        'score': dados[:, 1],
        'idade': dados[:, 2],
        'label': labels_justos,
        'genero': genero
    }),
    label_names=['label'],
    protected_attribute_names=['genero'],
    favorable_label=1,
    unfavorable_label=0
)

# Calcular métricas de justiça pré-processamento
metric_original = BinaryLabelDatasetMetric(
    dataset,
    unprivileged_groups=[{'genero': 0}],  # Grupo desprivilegiado: mulheres
    privileged_groups=[{'genero': 1}]     # Grupo privilegiado: homens
)

print("=== ANÁLISE DE VIÉS COM AIF360 ===")
print(f"Taxa de aprovação mulheres: {metric_original.disparate_impact():.2%}")
print(f"Diferença estatística: {metric_original.statistical_parity_difference():.3f}")
print(f"Disparate Impact: {metric_original.disparate_impact():.3f}")

# Aplicar reamostragem para reduzir viés
RW = Reweighing(unprivileged_groups=[{'genero': 0}],
                privileged_groups=[{'genero': 1}])
dataset_transf = RW.fit_transform(dataset)

# Verificar melhoria após reamostragem
metric_transf = BinaryLabelDatasetMetric(
    dataset_transf,
    unprivileged_groups=[{'genero': 0}],
    privileged_groups=[{'genero': 1}]
)

print("\n=== APÓS CORREÇÃO DE VIÉS ===")
print(f"Disparate Impact corrigido: {metric_transf.disparate_impact():.3f}")
print(f"Diferença estatística corrigida: {metric_transf.statistical_parity_difference():.3f}")

if abs(metric_transf.statistical_parity_difference()) < 0.1:
    print("✅ Viés reduzido significativamente!")
else:
    print("⚠️  Ainda há viés significativo")

=== ANÁLISE DE VIÉS COM AIF360 ===
Taxa de aprovação mulheres: 56.79%
Diferença estatística: -0.254
Disparate Impact: 0.568

=== APÓS CORREÇÃO DE VIÉS ===
Disparate Impact corrigido: 1.000
Diferença estatística corrigida: 0.000
✅ Viés reduzido significativamente!


##Algoritmo usando FairLearn (Microsoft) - Mitigação de Viés em Admissões

In [ ]:
!pip install fairlearn

In [ ]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from fairlearn.metrics import demographic_parity_difference, equalized_odds_difference
from fairlearn.reductions import ExponentiatedGradient, DemographicParity

# Simular dados de admissão universitária
np.random.seed(123)
n = 300

# Criar dados sintéticos
dados = pd.DataFrame({
    'nota_enem': np.random.normal(700, 100, n),
    'renda_familiar': np.random.exponential(3000, n),
    'escola_publica': np.random.binomial(1, 0.6, n),
    'raca': np.random.choice([0, 1], n, p=[0.3, 0.7]),  # 0=minoria, 1=maioria
    'extracurriculares': np.random.randint(0, 5, n)
})

# Criar labels com viés sistêmico (minoria tem 20% menos chance)
viés = np.where(dados['raca'] == 0, -0.2, 0)
probabilidade = 1 / (1 + np.exp(-(
    0.01 * dados['nota_enem'] +
    0.0001 * dados['renda_familiar'] +
    0.3 * dados['escola_publica'] +
    0.2 * dados['extracurriculares'] +
    viés + np.random.normal(0, 0.5, n) - 8.0 # Added -8.0 to shift probabilities
)))

dados['admitido'] = (probabilidade > 0.5).astype(int)

# Separar features e labels
X = dados[['nota_enem', 'renda_familiar', 'escola_publica', 'extracurriculares']]
y = dados['admitido']
sensitive_features = dados['raca']

print("=== ANÁLISE DE JUSTIÇA COM FAIRLEARN ===")

# Treinar modelo base (com viés)
modelo_base = LogisticRegression(solver='liblinear') # Added solver for older sklearn versions
modelo_base.fit(X, y)
pred_base = modelo_base.predict(X)

# Calcular métricas de justiça
dp_diff_base = demographic_parity_difference(
    y_true=y,
    y_pred=pred_base,
    sensitive_features=sensitive_features
)

eod_diff_base = equalized_odds_difference(
    y_true=y,
    y_pred=pred_base,
    sensitive_features=sensitive_features
)

print("\n--- Modelo Base (sem mitigação) ---")
print(f"Demographic Parity Difference: {dp_diff_base:.3f}")
print(f"Equalized Odds Difference: {eod_diff_base:.3f}")
print(f"Acurácia: {(pred_base == y).mean():.2%}")

# Aplicar mitigação de viés usando Exponentiated Gradient
constraint = DemographicParity()
modelo_justo = ExponentiatedGradient(
    estimator=LogisticRegression(solver='liblinear'), # Added solver
    constraints=constraint
)

modelo_justo.fit(X, y, sensitive_features=sensitive_features)
pred_justo = modelo_justo.predict(X)

# Calcular métricas após mitigação
dp_diff_justo = demographic_parity_difference(
    y_true=y,
    y_pred=pred_justo,
    sensitive_features=sensitive_features
)

eod_diff_justo = equalized_odds_difference(
    y_true=y,
    y_pred=pred_justo,
    sensitive_features=sensitive_features
)

print("\n--- Modelo com Mitigação de Viés ---")
print(f"Demographic Parity Difference: {dp_diff_justo:.3f}")
print(f"Equalized Odds Difference: {eod_diff_justo:.3f}")
print(f"Acurácia: {(pred_justo == y).mean():.2%}")

# Análise comparativa
print("\n=== RESUMO DA MITIGAÇÃO ===")
# Check if dp_diff_base is not zero to avoid division by zero
if dp_diff_base != 0:
    print(f"Redução no viés (Demographic Parity): {(dp_diff_base - dp_diff_justo)/dp_diff_base:.1%}")
else:
    print("Redução no viés (Demographic Parity): Não aplicável (viés base zero)")

# Check if eod_diff_base is not zero to avoid division by zero
if eod_diff_base != 0:
    print(f"Redução no viés (Equalized Odds): {(eod_diff_base - eod_diff_justo)/eod_diff_base:.1%}")
else:
    print("Redução no viés (Equalized Odds): Não aplicável (viés base zero)")

if dp_diff_justo < 0.1 and eod_diff_justo < 0.1:
    print("✅ Modelo atende aos critérios de justiça!")
else:
    print("⚠️  Modelo ainda apresenta viés significativo")

=== ANÁLISE DE JUSTIÇA COM FAIRLEARN ===

--- Modelo Base (sem mitigação) ---
Demographic Parity Difference: 0.023
Equalized Odds Difference: 0.015
Acurácia: 86.00%

--- Modelo com Mitigação de Viés ---
Demographic Parity Difference: 0.010
Equalized Odds Difference: 0.064
Acurácia: 86.67%

=== RESUMO DA MITIGAÇÃO ===
Redução no viés (Demographic Parity): 58.3%
Redução no viés (Equalized Odds): -324.0%
✅ Modelo atende aos critérios de justiça!


## What-If Tool: Propósito, Diferenciais e Desafio!

Embora não tenhamos conseguido executar diretamente a **What-If Tool (WIT)** neste ambiente Colab devido a conflitos de dependência de bibliotecas, é crucial entender sua importância e diferenciais para a análise de fairness em modelos de IA.

### O que é a What-If Tool e para que serve?

A What-If Tool é uma ferramenta interativa e visual, desenvolvida pelo Google, que permite aos desenvolvedores e analistas de IA explorar modelos de aprendizado de máquina de forma simples, sem a necessidade de escrever código. Seu principal objetivo é ajudar a entender o comportamento do modelo em diferentes cenários, especialmente em relação a questões de fairness e desempenho.

Com a WIT, você pode:

*   **Testar cenários hipotéticos:** Altere um ou mais pontos de dados e veja como o modelo reage, ou seja, como as previsões mudam. Isso é o que tentamos simular manualmente.
*   **Visualizar dados e resultados do modelo:** Observe a distribuição das previsões e das características de entrada por diferentes grupos (ex: gênero, raça, região).
*   **Analisar métricas de fairness:** Calcule e visualize métricas como `taxas de falsos positivos/negativos`, `disparidade demográfica` e `igualdade de chances` entre grupos definidos, ajudando a identificar vieses.
*   **Explorar a robustez do modelo:** Entenda como o modelo se comporta com pequenas perturbações nos dados.

### Diferenciais da What-If Tool em comparação com a simulação manual:

Nossa simulação manual, embora eficaz para demonstrar o conceito do "What-If" para um indivíduo específico, tem limitações significativas em comparação com a WIT:

1.  **Interatividade e Visualização:** A WIT oferece uma interface gráfica rica e interativa. Você pode explorar dados, mudar características e ver os resultados instantaneamente através de gráficos, sem a necessidade de codificar cada cenário.
2.  **Análise Agregada de Fairness:** Enquanto nossa simulação foca em um indivíduo, a WIT permite analisar padrões de viés em grandes conjuntos de dados e grupos, revelando se um modelo está se comportando de forma diferente (e potencialmente injusta) para categorias inteiras de usuários.
3.  **Exploração de Contrafactuais:** A ferramenta facilita a criação de exemplos contrafactuais, ou seja, pergunta "o que teria acontecido se...?" para muitos exemplos ao mesmo tempo, buscando o menor conjunto de mudanças nos atributos que alteraria a previsão do modelo.
4.  **Integração com ecossistemas de ML:** Ela se integra bem com modelos treinados em TensorFlow, PyTorch e scikit-learn, oferecendo uma experiência mais fluida.

### Conclusão e Convite para o Futuro:

A incompatibilidade da `witwidget` com a versão atual do `protobuf` no ambiente Colab impediu a utilização plena desta poderosa ferramenta. As dependências conflitantes tornaram inviável sua execução sem comprometer outras bibliotecas essenciais para o projeto (16/12/25).

No entanto, este exercício de simulação manual foi fundamental para entender o conceito de "What-If" e o impacto dos atributos sensíveis nas decisões do modelo.

**Convidamos as próximas turmas a considerar a execução deste notebook em plataformas que possam suportar as dependências da What-If Tool sem conflitos (como um ambiente local com versões específicas de bibliotecas, ou outras plataformas de nuvem/ML que ofereçam mais controle sobre o ambiente). Esta seria uma excelente melhoria futura para o projeto, permitindo uma análise muito mais rica e aprofundada da fairness em IA.**